In [9]:
from pathlib import Path
import sqlite3
import pandas as pd
from scipy import stats

In [ ]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == "python" else Path.cwd()

db_path = BASE_DIR / "data" / "processed" / "supply_chain.db"
export_path = BASE_DIR / "data" / "processed" / "pbi_sc_export.csv"

In [11]:
conn = sqlite3.connect(db_path)

query = """
SELECT 
    vw.*, 
    o.unit_quant
FROM vw_order_economics AS vw
JOIN order_list AS o ON vw.order_id = o.order_id;
"""

df = pd.read_sql_query(query, conn)

conn.close()

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9634 entries, 0 to 9633
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          9634 non-null   object 
 1   carrier           9634 non-null   object 
 2   service_level     9634 non-null   object 
 3   orig_port         9634 non-null   object 
 4   dest_port         9634 non-null   object 
 5   freight_cost      9634 non-null   float64
 6   warehousing_cost  9634 non-null   float64
 7   total_cost        9634 non-null   float64
 8   is_vmi            9634 non-null   int64  
 9   unit_quant        9634 non-null   int64  
dtypes: float64(3), int64(2), object(5)
memory usage: 752.8+ KB


In [13]:
df['cost_per_unit'] = df['total_cost']/ df['unit_quant']
vmi_costs = df[df['is_vmi'] == 1]['cost_per_unit']
non_vmi_costs = df[df['is_vmi'] == 0]['cost_per_unit']

In [14]:
stats.ttest_ind(vmi_costs, non_vmi_costs, equal_var=False)

TtestResult(statistic=np.float64(-2.5037921948589656), pvalue=np.float64(0.012308667580652907), df=np.float64(7300.484233881768))

In [15]:
print("Freight Sum:", df['freight_cost'].sum())
print("VMI Value Counts:\n", df['is_vmi'].value_counts())

Freight Sum: 80874.3828734672
VMI Value Counts:
 is_vmi
0    7126
1    2508
Name: count, dtype: int64


In [16]:
df.to_csv(export_path, index = False)